In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_ROOT = '/content/drive/MyDrive/ipynb-project/Road-Damage-Severity-Levels'
os.chdir(PROJECT_ROOT)

print(f"Working directory: {os.getcwd()}")

Mounted at /content/drive
Working directory: /content/drive/MyDrive/ipynb-project/Road-Damage-Severity-Levels


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
from collections import Counter
import random
from tqdm import tqdm

random.seed(42)

print("Libraries imported")

Libraries imported


In [ ]:
def safe_save_path(path):
    """Ensure directory exists in Google Drive before use"""
    os.makedirs(path, exist_ok=True)
    return path

# Paths
PROCESSED_DATASET = safe_save_path(os.path.join(PROJECT_ROOT, 'processed_dataset'))
VALIDATION_DIR = safe_save_path(os.path.join(PROJECT_ROOT, 'validation_samples'))
RESULTS_DIR = safe_save_path(os.path.join(PROJECT_ROOT, 'analysis_results'))

# Load severity thresholds
threshold_file = os.path.join(RESULTS_DIR, 'severity_thresholds.json')
with open(threshold_file, 'r') as f:
    SEVERITY_THRESHOLDS = json.load(f)

SEVERITY_THRESHOLDS = {int(k): v for k, v in SEVERITY_THRESHOLDS.items()}

# Class config
CLASS_NAMES = {
    0: 'Longitudinal Crack',
    1: 'Transverse Crack',
    2: 'Alligator Crack',
    4: 'Pothole'
}

USED_CLASSES = [0, 1, 2, 4]
SEVERITY_LABELS = ['Rendah', 'Sedang', 'Tinggi']

print("Configuration loaded")
print(f"Processed dataset: {PROCESSED_DATASET}")
print(f"Validation output: {VALIDATION_DIR}")

Configuration loaded
Processed dataset: /content/drive/MyDrive/ipynb-project/Road-Damage-Severity-Levels/processed_dataset
Validation output: /content/drive/MyDrive/ipynb-project/Road-Damage-Severity-Levels/validation_samples


In [ ]:
def load_severity_annotation(json_path):
    """Load severity annotation from JSON file"""
    with open(json_path, 'r') as f:
        return json.load(f)

def parse_yolo_label(label_path):
    """Parse YOLO format label file"""
    bboxes = []

    if not os.path.exists(label_path):
        return bboxes

    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue

            class_id = int(parts[0])
            x_center = float(parts[1])
            y_center = float(parts[2])
            width = float(parts[3])
            height = float(parts[4])
            area_ratio = width * height

            bboxes.append({
                'class_id': class_id,
                'x_center': x_center,
                'y_center': y_center,
                'width': width,
                'height': height,
                'area_ratio': area_ratio  # ← TAMBAHKAN INI
            })

    return bboxes

print("Helper functions defined")

Helper functions defined


In [ ]:
print("Selecting validation samples...")

# Sample 50 images per class (200 total)
SAMPLES_PER_CLASS = 50

train_severity_dir = os.path.join(PROCESSED_DATASET, 'train', 'severity')
all_severity_files = [f for f in os.listdir(train_severity_dir) if f.endswith('.json')]

# Organize by class
images_by_class = {c: [] for c in USED_CLASSES}

for sev_file in all_severity_files:
    sev_path = os.path.join(train_severity_dir, sev_file)
    data = load_severity_annotation(sev_path)

    # Get classes in this image
    classes_in_image = set([ann['class_id'] for ann in data['annotations']])

    for class_id in classes_in_image:
        if class_id in USED_CLASSES:
            images_by_class[class_id].append(data['image_name'])

print("\nAvailable images per class:")
for class_id in USED_CLASSES:
    print(f"  Class {class_id} ({CLASS_NAMES[class_id]}): {len(images_by_class[class_id])} images")

# Sample stratified
selected_samples = {}
for class_id in USED_CLASSES:
    available = list(set(images_by_class[class_id]))  # Remove duplicates
    n_samples = min(SAMPLES_PER_CLASS, len(available))
    selected_samples[class_id] = random.sample(available, n_samples)
    print(f"  Selected {n_samples} samples for class {class_id}")

# Flatten to unique list
all_selected = set()
for samples in selected_samples.values():
    all_selected.update(samples)

all_selected = list(all_selected)
print(f"\nTotal unique images selected for validation: {len(all_selected)}")

Selecting validation samples...

Available images per class:
  Class 0 (Longitudinal Crack): 5202 images
  Class 1 (Transverse Crack): 3284 images
  Class 2 (Alligator Crack): 3362 images
  Class 4 (Pothole): 1574 images
  Selected 50 samples for class 0
  Selected 50 samples for class 1
  Selected 50 samples for class 2
  Selected 50 samples for class 4

Total unique images selected for validation: 199


In [ ]:
def visualize_validation_sample(img_name, save_dir):
    """Create visualization with bbox and severity labels"""

    # Paths
    img_path = os.path.join(PROCESSED_DATASET, 'train', 'images', img_name)
    lbl_path = os.path.join(PROCESSED_DATASET, 'train', 'labels',
                           os.path.splitext(img_name)[0] + '.txt')
    sev_path = os.path.join(PROCESSED_DATASET, 'train', 'severity',
                           os.path.splitext(img_name)[0] + '.json')

    # Load data
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    bboxes = parse_yolo_label(lbl_path)
    sev_data = load_severity_annotation(sev_path)

    # Color mapping for severity
    severity_colors = {
        0: (0, 255, 0),      # Rendah = Green
        1: (255, 165, 0),    # Sedang = Orange
        2: (255, 0, 0)       # Tinggi = Red
    }

    # Draw bboxes with severity
    for bbox, sev_ann in zip(bboxes, sev_data['annotations']):
        x_center = int(bbox['x_center'] * w)
        y_center = int(bbox['y_center'] * h)
        box_w = int(bbox['width'] * w)
        box_h = int(bbox['height'] * h)

        x1 = x_center - box_w // 2
        y1 = y_center - box_h // 2
        x2 = x_center + box_w // 2
        y2 = y_center + box_h // 2

        severity = sev_ann['severity']
        color = severity_colors[severity]
        bbox_id = sev_ann['bbox_id']

        # Draw bbox with THINNER line (2 instead of 3)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

        # Draw label with bbox_id
        label = f"#{bbox_id} C{bbox['class_id']}-{SEVERITY_LABELS[severity]}"
        label_bg_y1 = max(0, y1 - 25)
        cv2.rectangle(img, (x1, label_bg_y1), (x1 + 180, y1), color, -1)
        cv2.putText(img, label, (x1 + 5, y1 - 8),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    # Save
    output_path = os.path.join(save_dir, img_name)
    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    cv2.imwrite(output_path, img_bgr)

    return sev_data

print("Generating validation visualizations...")

# Create subdirectories per class
for class_id in USED_CLASSES:
    safe_save_path(os.path.join(VALIDATION_DIR, f'class_{class_id}'))

validation_metadata = []
processed_images = set()

for class_id in USED_CLASSES:
    class_dir = os.path.join(VALIDATION_DIR, f'class_{class_id}')

    print(f"\nProcessing Class {class_id} ({CLASS_NAMES[class_id]})...")

    for img_name in tqdm(selected_samples[class_id]):
        # Visualize (save to class folder)
        sev_data = visualize_validation_sample(img_name, class_dir)

        if img_name not in processed_images:
            # Ambil SEMUA annotations, bukan hanya yang sesuai class_id
            for ann in sev_data['annotations']:
                # Get bbox coordinates for identification
                bbox_coords = ann['bbox']
                bbox_x = round(bbox_coords[0], 3)  # x_center
                bbox_y = round(bbox_coords[1], 3)  # y_center
                bbox_w = round(bbox_coords[2], 3)  # width
                bbox_h = round(bbox_coords[3], 3)  # height

                validation_metadata.append({
                    'image_name': img_name,
                    'bbox_id': ann['bbox_id'],
                    'bbox_center_x': bbox_x,
                    'bbox_center_y': bbox_y,
                    'bbox_width': bbox_w,
                    'bbox_height': bbox_h,
                    'class_id': ann['class_id'],
                    'class_name': CLASS_NAMES[ann['class_id']],
                    'severity': ann['severity'],
                    'severity_label': ann['severity_label'],
                    'area_ratio': ann['area_ratio'],
                    'auto_assigned': True,
                    'manual_review': None,
                    'notes': ''
                })

            processed_images.add(img_name)

print(f"\nVisualization complete")
print(f"Total unique images processed: {len(processed_images)}")
print(f"Total annotations generated: {len(validation_metadata)}")

Generating validation visualizations...

Processing Class 0 (Longitudinal Crack)...


100%|██████████| 50/50 [01:18<00:00,  1.56s/it]



Processing Class 1 (Transverse Crack)...


100%|██████████| 50/50 [01:15<00:00,  1.51s/it]



Processing Class 2 (Alligator Crack)...


100%|██████████| 50/50 [01:09<00:00,  1.39s/it]



Processing Class 4 (Pothole)...


100%|██████████| 50/50 [01:12<00:00,  1.46s/it]


Visualization complete
Total unique images processed: 199
Total annotations generated: 673


In [ ]:
# Create DataFrame
df_validation = pd.DataFrame(validation_metadata)

# Add empty columns for manual review
df_validation['manual_severity'] = ''
df_validation['is_correct'] = ''
df_validation['reviewer_notes'] = ''

# Save to CSV
checklist_path = os.path.join(VALIDATION_DIR, 'validation_checklist.csv')
df_validation.to_csv(checklist_path, index=False)

print(f"Validation checklist created:")
print(f"   {checklist_path}")
print(f"\nTotal annotations to review: {len(df_validation)}")

# Show distribution
print("\nDistribution by class and severity:")
dist = df_validation.groupby(['class_id', 'severity_label']).size().unstack(fill_value=0)
print(dist)

Validation checklist created:
   /content/drive/MyDrive/ipynb-project/Road-Damage-Severity-Levels/validation_samples/validation_checklist.csv

Total annotations to review: 673

Distribution by class and severity:
severity_label  Rendah  Sedang  Tinggi
class_id                              
0                  227      20       0
1                  151       3       1
2                   72      40      30
4                  124       4       1


In [ ]:
instructions = """
╔══════════════════════════════════════════════════════════════════════════════╗
║                                                                              ║
║          PANDUAN VALIDASI MANUAL SEVERITY ANNOTATION                        ║
║          Klasifikasi Tingkat Bahaya Kerusakan Jalan                         ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝

 LOKASI FILE VALIDASI
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Folder: {validation_dir}

Struktur:
   class_0/  → Longitudinal Crack (Retak Memanjang)
   class_1/  → Transverse Crack (Retak Melintang)
   class_2/  → Alligator Crack (Retak Kulit Buaya)
   class_4/  → Pothole (Lubang)

   validation_checklist.csv  ← FILE YANG HARUS DIISI


══════════════════════════════════════════════════════════════════════════════
 TUJUAN VALIDASI
═══════════════════════════════════════════════════════════════════════════════

Memvalidasi apakah threshold severity otomatis (yang dihitung dari area ratio
bounding box) sudah sesuai dengan penilaian visual manusia.


═══════════════════════════════════════════════════════════════════════════════
 KRITERIA SEVERITY - BERDASARKAN AREA KERUSAKAN DALAM FRAME
═══════════════════════════════════════════════════════════════════════════════

┌─────────────────────────────────────────────────────────────────────────────┐
│  RENDAH (Low Severity)                                                    │
├─────────────────────────────────────────────────────────────────────────────┤
│ • Area kerusakan: KURANG DARI 5% total frame                               │
│ • Visualisasi: Kerusakan terlihat kecil/tipis relatif terhadap foto        │
│ • Contoh:                                                                   │
│   - Retak tipis seperti garis pensil                                        │
│   - Lubang kecil seukuran koin                                              │
│   - Kerusakan yang hanya memakan pojok/sudut kecil frame                    │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────┐
│  SEDANG (Medium Severity)                                                 │
├─────────────────────────────────────────────────────────────────────────────┤
│ • Area kerusakan: ANTARA 5% - 15% total frame                              │
│ • Visualisasi: Kerusakan jelas terlihat, memakan area signifikan           │
│ • Contoh:                                                                   │
│   - Retak yang cukup lebar/panjang, tapi tidak dominan                      │
│   - Lubang berukuran sedang                                                 │
│   - Kerusakan yang memakan sekitar 1/10 hingga 1/6 area frame              │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────┐
│  TINGGI (High Severity)                                                   │
├─────────────────────────────────────────────────────────────────────────────┤
│ • Area kerusakan: LEBIH DARI 15% total frame                               │
│ • Visualisasi: Kerusakan sangat dominan, memakan area besar foto           │
│ • Contoh:                                                                   │
│   - Alligator crack yang menyebar luas                                      │
│   - Lubang besar yang sangat terlihat                                       │
│   - Kerusakan yang memakan lebih dari 1/6 area frame                        │
└─────────────────────────────────────────────────────────────────────────────┘


═══════════════════════════════════════════════════════════════════════════════
 RULE OF THUMB - CARA CEPAT MENILAI
═══════════════════════════════════════════════════════════════════════════════

Ketika melihat gambar, tanyakan pada diri Anda:

┌─────────────────────────────────────────────────────────────────────────────┐
│                                                                             │
│  "Jika frame foto ini dibagi 20 kotak sama besar (4×5),                    │
│   berapa kotak yang tertutupi kerusakan?"                                  │
│                                                                             │
│   • Kurang dari 1 kotak        → RENDAH                                    │
│   • 1-3 kotak                   → SEDANG                                    │
│   • Lebih dari 3 kotak          → TINGGI                                    │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘

ATAU gunakan acuan ini:

   RENDAH   = Kerusakan lebih kecil dari 1 jari telunjuk di layar Anda
   SEDANG   = Kerusakan seukuran 2-3 jari atau setengah telapak tangan
   TINGGI   = Kerusakan lebih besar dari telapak tangan


═══════════════════════════════════════════════════════════════════════════════
CARA MENGISI VALIDATION_CHECKLIST.CSV
═══════════════════════════════════════════════════════════════════════════════

Buka file CSV di Google Sheets (JANGAN buka langsung, tapi IMPORT dulu!):
  1. Download file validation_checklist.csv
  2. Buka Google Sheets baru
  3. File → Import → Upload → Pilih validation_checklist.csv
  4. Separator type: Comma
  5. Klik "Import data"

Isi 3 kolom berikut untuk setiap baris:

┌─────────────────────────┬────────────────────────────────────────────────────┐
│ Kolom                   │ Cara Mengisi                                       │
├─────────────────────────┼────────────────────────────────────────────────────┤
│ manual_severity         │ Tulis: Rendah / Sedang / Tinggi                   │
│                         │ (sesuai penilaian Anda setelah lihat gambar)      │
├─────────────────────────┼────────────────────────────────────────────────────┤
│ is_correct              │ Tulis: Yes / No                                    │
│                         │ Yes = Anda setuju dengan label otomatis           │
│                         │ No = Anda tidak setuju                             │
├─────────────────────────┼────────────────────────────────────────────────────┤
│ reviewer_notes          │ (Opsional) Tulis alasan jika pilih "No"           │
│                         │ Contoh: "Area terlalu kecil untuk Sedang"         │
└─────────────────────────┴────────────────────────────────────────────────────┘


═══════════════════════════════════════════════════════════════════════════════
SKENARIO KHUSUS & CARA MENGATASINYA
═══════════════════════════════════════════════════════════════════════════════

┌─────────────────────────────────────────────────────────────────────────────┐
│ SKENARIO 1: Saya bingung ini berapa persen area-nya                        │
├─────────────────────────────────────────────────────────────────────────────┤
│ SOLUSI:                                                                     │
│ → Lihat kolom 'area_ratio' di CSV sebagai referensi                        │
│ → area_ratio = 0.04 berarti 4% dari frame → RENDAH                         │
│ → area_ratio = 0.10 berarti 10% dari frame → SEDANG                        │
│ → area_ratio = 0.20 berarti 20% dari frame → TINGGI                        │
│                                                                             │
│ TAPI: Jangan hanya copy nilai area_ratio! Tetap lihat gambar dulu.         │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────┐
│ SKENARIO 2: Foto dari jauh, kerusakan kecil tapi saya tahu itu berbahaya   │
├─────────────────────────────────────────────────────────────────────────────┤
│ SOLUSI:                                                                     │
│ → Tetap nilai berdasarkan AREA VISUAL di frame, bukan bahaya nyata         │
│ → Jika area <5% frame → pilih RENDAH (meskipun Anda tahu itu berbahaya)    │
│                                                                             │
│ ALASAN:                                                                     │
│ → Model hanya bisa lihat pixel 2D, tidak bisa tahu kedalaman               │
│ → Kita validasi apakah threshold area_ratio sudah konsisten visual         │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────┐
│ SKENARIO 3: Area 5-15% (border antara Rendah-Sedang atau Sedang-Tinggi)    │
├─────────────────────────────────────────────────────────────────────────────┤
│ SOLUSI:                                                                     │
│ → Jika ragu antara Rendah vs Sedang: Pilih yang lebih mendekati           │
│ → Area 4.8% → RENDAH (lebih dekat ke batas bawah)                          │
│ → Area 5.2% → SEDANG (sudah masuk threshold)                               │
│                                                                             │
│ → Jika benar-benar 50-50, cek area_ratio:                                  │
│   • area_ratio < 0.05 → RENDAH                                             │
│   • area_ratio 0.05-0.15 → SEDANG                                          │
│   • area_ratio > 0.15 → TINGGI                                             │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────┐
│ SKENARIO 4: Satu gambar ada banyak bounding box dengan severity berbeda    │
├─────────────────────────────────────────────────────────────────────────────┤
│ SOLUSI:                                                                     │
│ → Nilai SETIAP bounding box secara INDEPENDEN                              │
│ → Dalam 1 gambar bisa ada bbox Rendah, Sedang, Tinggi sekaligus           │
│ → Lihat label di pojok bbox untuk tahu mana yang sedang dinilai            │
└─────────────────────────────────────────────────────────────────────────────┘


═══════════════════════════════════════════════════════════════════════════════
CHECKLIST SEBELUM SUBMIT
═══════════════════════════════════════════════════════════════════════════════

Sebelum menyerahkan hasil validasi, pastikan:

  ☐ Semua 200+ baris sudah diisi (tidak ada yang kosong di kolom wajib)
  ☐ Kolom 'manual_severity' hanya berisi: Rendah / Sedang / Tinggi
  ☐ Kolom 'is_correct' hanya berisi: Yes / No
  ☐ File disimpan sebagai .csv (bukan .xlsx)
  ☐ Minimal 2 orang sudah review untuk cross-check


═══════════════════════════════════════════════════════════════════════════════
TARGET VALIDASI
═══════════════════════════════════════════════════════════════════════════════

Setelah review selesai, hitung:

  Persentase "is_correct" = Yes: __________%

  • Jika ≥ 70% → Threshold bagus, lanjut ke training
  • Jika < 70% → Threshold perlu disesuaikan


═══════════════════════════════════════════════════════════════════════════════
PENTING: PRINSIP UTAMA
═══════════════════════════════════════════════════════════════════════════════

  1. FOKUS PADA AREA VISUAL dalam frame, BUKAN bahaya nyata di lapangan

  2. jika terlihat kecil, maka Rendah; besar, maka Tinggi

  3. Gunakan area_ratio sebagai REFERENSI, bukan sebagai jawaban final

  4. Jika ragu, gunakan Rule of Thumb "20 kotak" atau "ukuran jari"

  5. Konsisten lebih penting daripada sempurna


═══════════════════════════════════════════════════════════════════════════════

""".format(validation_dir=VALIDATION_DIR)

# Save instructions
instructions_path = os.path.join(VALIDATION_DIR, 'VALIDATION_INSTRUCTIONS.txt')
with open(instructions_path, 'w', encoding='utf-8') as f:
    f.write(instructions)

print("✅ Validation instructions created:")
print(f"   {instructions_path}")
print("\n" + "="*80)
print(instructions)

✅ Validation instructions created:
   /content/drive/MyDrive/ipynb-project/Road-Damage-Severity-Levels/validation_samples/VALIDATION_INSTRUCTIONS.txt


╔══════════════════════════════════════════════════════════════════════════════╗
║                                                                              ║
║          PANDUAN VALIDASI MANUAL SEVERITY ANNOTATION                        ║
║          Klasifikasi Tingkat Bahaya Kerusakan Jalan                         ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝

 LOKASI FILE VALIDASI
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Folder: /content/drive/MyDrive/ipynb-project/Road-Damage-Severity-Levels/validation_samples

Struktur:
   class_0/  → Longitudinal Crack (Retak Memanjang)
   class_1/  → Transverse Crack (Retak Melintang)
   class_2/  → Alligator Crack (Retak Kulit Buaya)
   cla

In [ ]:
summary_report = {
    'total_images': len(all_selected),
    'total_annotations': len(validation_metadata),
    'samples_per_class': {int(k): len(v) for k, v in selected_samples.items()},
    'severity_distribution': df_validation.groupby('severity_label').size().to_dict(),
    'validation_checklist_path': checklist_path,
    'instructions_path': instructions_path,
    'validation_images_dir': VALIDATION_DIR
}

# Save summary
summary_path = os.path.join(VALIDATION_DIR, 'validation_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary_report, f, indent=2)

print("\n")
print("VALIDATION SUMMARY")
print("="*60)
print(f"Total images: {summary_report['total_images']}")
print(f"Total annotations: {summary_report['total_annotations']}")
print(f"\nSamples per class:")
for class_id, count in summary_report['samples_per_class'].items():
    print(f"  Class {class_id}: {count} images")
print(f"\nSeverity distribution:")
for sev, count in summary_report['severity_distribution'].items():
    print(f"  {sev}: {count} annotations")

print(f"\nSummary saved to: {summary_path}")



VALIDATION SUMMARY
Total images: 199
Total annotations: 673

Samples per class:
  Class 0: 50 images
  Class 1: 50 images
  Class 2: 50 images
  Class 4: 50 images

Severity distribution:
  Rendah: 574 annotations
  Sedang: 67 annotations
  Tinggi: 32 annotations

Summary saved to: /content/drive/MyDrive/ipynb-project/Road-Damage-Severity-Levels/validation_samples/validation_summary.json


In [ ]:
print("\n")
print("NOTEBOOK 3 COMPLETE")
print("="*60)
print("\nGenerated files:")
print(f"   {VALIDATION_DIR}/")
print("     ├── class_0/ (Longitudinal Crack samples)")
print("     ├── class_1/ (Transverse Crack samples)")
print("     ├── class_2/ (Alligator Crack samples)")
print("     ├── class_4/ (Pothole samples)")
print("     ├── validation_checklist.csv FILL THIS")
print("     ├── VALIDATION_INSTRUCTIONS.txt")
print("     └── validation_summary.json")



NOTEBOOK 3 COMPLETE

Generated files:
   /content/drive/MyDrive/ipynb-project/Road-Damage-Severity-Levels/validation_samples/
     ├── class_0/ (Longitudinal Crack samples)
     ├── class_1/ (Transverse Crack samples)
     ├── class_2/ (Alligator Crack samples)
     ├── class_4/ (Pothole samples)
     ├── validation_checklist.csv FILL THIS
     ├── VALIDATION_INSTRUCTIONS.txt
     └── validation_summary.json


In [ ]:
print("\nSyncing all files to Google Drive...")
from google.colab import drive
drive.flush_and_unmount()
print(" files synced successfully!")
print("If you want to continue working, re-run Cell 1 to mount Drive again.")


Syncing all files to Google Drive...
 files synced successfully!
If you want to continue working, re-run Cell 1 to mount Drive again.
